# QCDark2 SRDM Mediator Modes

This notebook validates `vector`, `scalar`, `approx`, and `approx_full` against `../QCDark2/qcdark2/dark_matter_rates.py::get_rate_flux` at the registered Si SRDM benchmark point.

## Mediator-mode reference section

In [ ]:
import numpy as np
from pathlib import Path
import sys

from DMeRates.data.registry import DataRegistry
from DMeRates.engines.dielectric import compute_dRdE

qcdark2_candidates = [
    Path("../QCDark2").resolve(),
    Path("../../QCDark2").resolve(),
]
qcdark2_repo = next((p for p in qcdark2_candidates if p.exists()), None)
if qcdark2_repo is None:
    raise FileNotFoundError("Could not locate ../QCDark2 or ../../QCDark2")
sys.path.insert(0, str(qcdark2_repo))
from qcdark2.dark_matter_rates import load_epsilon, get_rate_flux

MX_EV = 48232.9466
SIGMA_E_CM2 = 1.098541e-38
FDMN = 2
M_A_EV = 0.0
ENERGY_POINTS = [8.1, 14.9, 21.7]

flux_data = np.loadtxt(DataRegistry.srdm_flux_file("srdm_dphidv_DPLM_row10_col8.txt"), comments="#")
positive = flux_data[:, 0] > 0
v_list = flux_data[positive, 0] / 299792.458
flux = flux_data[positive, 1] * 299792.458

epsilon = load_epsilon(str(DataRegistry.qcdark2_dielectric("Si", "composite")))

qcdark2_names = {
    "vector": "vector",
    "scalar": "scalar",
    "approx": "approx",
    "approx_full": "approx full",
}


In [ ]:
refs = {}
summary = {}

for mode, qcdark2_mode in qcdark2_names.items():
    ref = get_rate_flux(
        epsilon,
        MX_EV,
        SIGMA_E_CM2,
        flux,
        v_list,
        m_A=M_A_EV,
        mediator=qcdark2_mode,
        screening="RPA",
    )
    ours = compute_dRdE(
        material="Si",
        mX_eV=MX_EV,
        sigma_e_cm2=SIGMA_E_CM2,
        FDMn=FDMN,
        mediator_spin=mode,
        halo_model="srdm",
        screening="rpa",
        variant="composite",
    )

    ours_vals = ours.dRdE_per_kg_per_year_per_eV
    pos = ref > 0
    rel = np.abs(ours_vals[pos] - ref[pos]) / ref[pos]
    summary[mode] = {
        "positive_bins": int(pos.sum()),
        "max_rel": float(rel.max()),
        "median_rel": float(np.median(rel)),
    }

    tuples = []
    for energy in ENERGY_POINTS:
        idx = int(np.argmin(np.abs(ours.E_eV - energy)))
        tuples.append((float(ours.E_eV[idx]), float(ref[idx])))
    refs[mode] = tuples

summary


In [ ]:
for mode, stats in summary.items():
    assert stats["max_rel"] < 0.05, (mode, stats)

refs
